# 02 · Entrenamiento de los modelos

Este notebook entrena la Regresión Logística y la Red Neuronal Artificial. Los pipelines completos se guardan en `models/` para que la aplicación Flask pueda utilizarlos sin volver a entrenar.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    MODEL_FEATURE_COLUMNS, TARGET_COLUMN, build_models, calculate_metrics,
    load_dataset, prepare_model_data
)

DATASET_PATH = PROJECT_ROOT / 'MASTER_CHART_F1000Research.xlsx'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
df = load_dataset(DATASET_PATH)
X = prepare_model_data(df)
y = (pd.to_numeric(df[TARGET_COLUMN]) == 2).astype(int)
print('X:', X.shape, '| y:', y.value_counts().to_dict())

X: (303, 13) | y: {0: 240, 1: 63}


## División estratificada

Se reserva el 20% de los casos para una evaluación independiente. El `random_state` permite reproducir los resultados.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Entrenamiento: {len(X_train)} filas')
print(f'Prueba: {len(X_test)} filas')

Entrenamiento: 242 filas
Prueba: 61 filas


## Construcción y entrenamiento

Cada pipeline contiene imputación por mediana y estandarización. La Regresión Logística usa balanceo de clases; la red neuronal tiene dos capas ocultas.

In [3]:
models = build_models()
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name}: entrenado')

logistic: entrenado
neural: entrenado


In [4]:
metrics = {}
for name, model in models.items():
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    metrics[name] = calculate_metrics(y_test, predictions, probabilities)

metrics_table = pd.DataFrame(metrics).T[[
    'accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc'
]]
metrics_table.style.format('{:.3f}')

,accuracy,precision,recall,specificity,f1,roc_auc
logistic,0.820,0.545,0.923,0.792,0.686,0.973
neural,0.770,0.400,0.154,0.938,0.222,0.808


## Guardado de artefactos

Se guardan los pipelines completos, incluyendo imputador y escalador. También se guardan los datos de prueba para que el notebook de evaluación y Flask calculen las mismas métricas.

In [5]:
joblib.dump(models['logistic'], MODELS_DIR / 'modelo_logistico.joblib')
joblib.dump(models['neural'], MODELS_DIR / 'modelo_neuronal.joblib')
joblib.dump({
    'x_test': X_test,
    'y_test': y_test,
    'feature_columns': MODEL_FEATURE_COLUMNS,
    'train_rows': len(X_train),
    'test_rows': len(X_test),
    'metrics': metrics,
}, MODELS_DIR / 'evaluation_data.joblib')
print('Modelos y evaluación guardados en:', MODELS_DIR.resolve())

Modelos y evaluación guardados en: C:\1-Universidad\6to\Analitica de datos\M3\models
